# NorthStar — MongoDB Atlas Design and PyMongo Development

**Notebook 5 of the analytical workflow.** Uses the cleaned data from `01_eda.ipynb`. The query-optimisation work that builds on this collection design lives in `06_indexing.ipynb`.

## Purpose
The relational analysis in notebooks 02–04 made the case for moving certain NorthStar data onto a document store. This notebook delivers that move:

1. Designs three MongoDB collections that match the operational reality of the business.
2. Loads them onto **MongoDB Atlas** (or any MongoDB instance — code is identical).
3. Demonstrates the full **CRUD** surface plus the **aggregation framework** that the analytics team will use day-to-day.

## Design rationale (the short version)

| Original CSV | Recommended target | Reason |
|---|---|---|
| `hubs`, `vehicles`, `drivers` | **Stay relational** | Stable, regular structure. Used as look-ups and join keys. No nesting. |
| `customers` | Becomes the *anchor* of `customer_cases` | Many child records per customer; one read gives the whole picture. |
| `orders`, `deliveries`, `incidents` | Embedded inside `customer_cases` | Tightly coupled to the customer and to each other; no separate analytical life. |
| `complaints` | New collection `complaint_cases` + embedded summary in `customer_cases` | Has its own resolution history with status-change events. The case study explicitly mentions *"long sequences of messages, repeated status changes, attachments, escalation notes"* — natural document shape. |
| `app_events` | New collection `app_sessions` | Session-scoped, variable-length, time-ordered. One document per session with an embedded `events` array. |

> **Marking alignment.** This notebook targets *MongoDB development* (20 marks): PyMongo, CRUD operations, NoSQL design, integration with Python. Sections 3 (design + load), 4 (read), 5 (update), 6 (delete) and 7 (aggregation) cover the full surface.


## 1. Setup and connection

The code below works against **MongoDB Atlas**

In [8]:
%pip install pymongo pandas python-dotenv -q

import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from pymongo import MongoClient

from google.colab import userdata
MONGO_URI = userdata.get("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client['northstar']
print('Connected. Existing collections:', db.list_collection_names())

Connected. Existing collections: []


In [9]:
# Load the cleaned CSVs from notebook 01
DATA_DIR = Path('cleaned_data')
assert DATA_DIR.exists(), f"Run 01_eda.ipynb first to produce {DATA_DIR}"

DATE_COLS = {
    'customers':   ['signup_date'],
    'vehicles':    ['commission_date'],
    'orders':      ['order_created_at'],
    'deliveries':  ['dispatch_time', 'delivery_completed_at'],
    'incidents':   ['reported_at'],
    'complaints':  ['created_at'],
    'app_events':  ['event_timestamp'],
}

tables = {}
for csv in sorted(DATA_DIR.glob('*.csv')):
    name = csv.stem
    tables[name] = pd.read_csv(csv, parse_dates=DATE_COLS.get(name))

for k, v in tables.items():
    print(f"{k:12s} {v.shape}")


app_events   (640, 10)
complaints   (320, 10)
customers    (650, 9)
deliveries   (950, 13)
drivers      (170, 8)
hubs         (8, 5)
incidents    (280, 7)
orders       (1250, 11)
vehicles     (120, 8)


## 2. Schema design — the three collections

Document modelling is not "tables minus the joins". It is about asking *what unit of information do I read and write together?* The answer drives the schema.

### 2.1 `app_sessions` — one document per session

A user's whole interaction with the mobile platform lives in one document, with the events as an embedded ordered array. The natural query — *"show me everything that happened in session S19847"* — becomes a single `find_one({'_id': 'S19847'})`. No joins. No flattening pain.

```
{
  _id: "S19847",            # session_id is the natural primary key
  customer_id: "C0488",
  device_type: "Android",
  zone_context: "North",
  session_start: ISODate(...),
  session_end:   ISODate(...),
  event_count: 7,
  events: [
    {event_type: "search_route", timestamp: ..., success_flag: 1, ...},
    {event_type: "track_order",  timestamp: ..., success_flag: 1, ...},
    ...
  ]
}
```

### 2.2 `complaint_cases` — one document per complaint with embedded resolution history

The case study explicitly mentions *"long sequences of messages, repeated status changes, attachments, escalation notes"*. That is a textbook document shape. The base data only contains the *summary* fields, so we **synthesise** a plausible `resolution_history` array from the existing `status`, `created_at`, and `resolution_days` fields — demonstrating the schema the production system would carry.

```
{
  _id: "CP0001",
  customer_id: "C0464", order_id: "O00814",
  complaint_type: "AppIssue", channel: "App", severity: "High",
  status: "Open", compensation_amount: 23.99,
  resolution_history: [
    {timestamp: ISODate(...), status: "Open",       actor: "System", note: "Complaint logged"},
    {timestamp: ISODate(...), status: "InProgress", actor: "Agent",  note: "Assigned to agent"},
    {timestamp: ISODate(...), status: "Resolved",   actor: "Agent",  note: "Resolved with compensation"}
  ]
}
```

### 2.3 `customer_cases` — the integrated operational view

This is the collection that delivers what the **board explicitly asked for**: *"an integrated operational view of customers, journeys, exceptions, complaints, and service events without the rigidity that currently limits reporting."*

One document per customer. Embeds every order they placed, with each order's delivery (if any) and that delivery's incidents nested inside. Embeds a list of complaint summaries. Includes a pre-computed `summary` block so dashboards don't have to recompute totals on every read.

```
{
  _id: "C0292",
  profile: {age: 35, home_zone: "North", customer_type: "SME", ...},
  orders: [
    {
      order_id: "O00001", service_type: "Passenger",
      order_value: 126.65, pickup_zone: "Airport", dropoff_zone: "South",
      delivery: {
        delivery_id: "DL00321", driver_id: "D004", hub_id: "H05",
        delivery_status: "Failed",
        customer_rating_post_delivery: 2.0,
        incidents: [{incident_id: "I0042", incident_type: "BatteryAlert", ...}]
      }
    },
    ...
  ],
  complaints: [{complaint_id: "CP0042", severity: "High", ...}],
  summary: {total_orders: 4, total_complaints: 1, total_value: 395.90}
}
```

**Why this is the central artefact.** A question that previously required a four-way join (`customers ⋈ orders ⋈ deliveries ⋈ complaints`) and a stitched-up application layer becomes a single `find_one({'_id': 'C0292'})`. The board's question — *"give me one place where I can see every fact about this customer"* — is answered structurally.

## 3. Build the three collections and load them

### 3.1 `app_sessions`

In [10]:
def to_session_doc(session_id, group):
    """Turn one session's app_events rows into a single document."""
    events_sorted = group.sort_values('event_timestamp')

    events = []
    for _, r in events_sorted.iterrows():
        events.append({
            'event_id':        r['event_id'],
            'event_type':      r['event_type'],
            'event_timestamp': r['event_timestamp'].to_pydatetime() if pd.notna(r['event_timestamp']) else None,
            'order_id':        r['order_id'] if pd.notna(r['order_id']) else None,
            'api_latency_ms':  int(r['api_latency_ms']) if pd.notna(r['api_latency_ms']) else None,
            'success_flag':    int(r['success_flag']),
        })

    return {
        '_id':           session_id,
        'customer_id':   group['customer_id'].mode().iloc[0] if not group['customer_id'].mode().empty else None,
        'device_type':   group['device_type'].mode().iloc[0],
        'zone_context':  group['zone_context'].mode().iloc[0],
        'session_start': events[0]['event_timestamp'],
        'session_end':   events[-1]['event_timestamp'],
        'event_count':   len(events),
        'events':        events,
    }

session_docs = [to_session_doc(sid, g)
                for sid, g in tables['app_events'].groupby('session_id')]

db['app_sessions'].drop()                           # idempotent
db['app_sessions'].insert_many(session_docs)
print(f"app_sessions: inserted {db['app_sessions'].count_documents({})} documents")
print("Sample document:")
import pprint; pprint.pp(db['app_sessions'].find_one(), width=80)


app_sessions: inserted 637 documents
Sample document:
{'_id': 'S10192',
 'customer_id': 'C0180',
 'device_type': 'Android',
 'zone_context': 'Central',
 'session_start': datetime.datetime(2025, 3, 19, 9, 20),
 'session_end': datetime.datetime(2025, 3, 19, 9, 20),
 'event_count': 1,
 'events': [{'event_id': 'AE00403',
             'event_type': 'payment_retry',
             'event_timestamp': datetime.datetime(2025, 3, 19, 9, 20),
             'order_id': 'O00149',
             'api_latency_ms': 108,
             'success_flag': 1}]}


### 3.2 `complaint_cases` — with synthesised resolution history

The base CSV only has `status`, `created_at`, `resolution_days`, `compensation_amount`. From those we can reconstruct the realistic resolution trajectory that the production system would store natively.

In [11]:
def synth_resolution_history(row):
    """Build a plausible resolution history from the summary fields we have."""
    events = [{
        'timestamp': row['created_at'].to_pydatetime(),
        'status': 'Open',
        'actor': 'System',
        'note': f"Complaint logged via {row['channel']}",
    }]
    if row['status'] == 'Resolved' and pd.notna(row['resolution_days']):
        half = int(row['resolution_days']) // 2
        events.append({
            'timestamp': (row['created_at'] + pd.Timedelta(days=half)).to_pydatetime(),
            'status': 'InProgress',
            'actor': 'Agent',
            'note': 'Assigned to support agent',
        })
        events.append({
            'timestamp': (row['created_at'] + pd.Timedelta(days=int(row['resolution_days']))).to_pydatetime(),
            'status': 'Resolved',
            'actor': 'Agent',
            'note': f"Resolved with compensation £{row.get('compensation_amount', 0):.2f}",
        })
    elif row['status'] == 'InProgress':
        events.append({
            'timestamp': (row['created_at'] + pd.Timedelta(days=1)).to_pydatetime(),
            'status': 'InProgress', 'actor': 'Agent', 'note': 'Under review',
        })
    return events


def to_complaint_doc(row):
    return {
        '_id':                 row['complaint_id'],
        'customer_id':         row['customer_id'],
        'order_id':            row['order_id'] if pd.notna(row['order_id']) else None,
        'complaint_type':      row['complaint_type'],
        'channel':             row['channel'],
        'severity':            row['severity'],
        'status':              row['status'],
        'compensation_amount': float(row['compensation_amount']) if pd.notna(row['compensation_amount']) else 0.0,
        'created_at':          row['created_at'].to_pydatetime(),
        'resolution_days':     float(row['resolution_days']) if pd.notna(row['resolution_days']) else None,
        'resolution_history':  synth_resolution_history(row),
    }


complaint_docs = [to_complaint_doc(r) for _, r in tables['complaints'].iterrows()]
db['complaint_cases'].drop()
db['complaint_cases'].insert_many(complaint_docs)
print(f"complaint_cases: inserted {db['complaint_cases'].count_documents({})} documents")
print("\nSample document:")
pprint.pp(db['complaint_cases'].find_one({'status':'Resolved'}), width=80)


complaint_cases: inserted 320 documents

Sample document:
{'_id': 'CP0005',
 'customer_id': 'C0535',
 'order_id': 'O00154',
 'complaint_type': 'Delay',
 'channel': 'Email',
 'severity': 'Medium',
 'status': 'Resolved',
 'compensation_amount': 16.18,
 'created_at': datetime.datetime(2024, 8, 31, 5, 56),
 'resolution_days': 1.0,
 'resolution_history': [{'timestamp': datetime.datetime(2024, 8, 31, 5, 56),
                         'status': 'Open',
                         'actor': 'System',
                         'note': 'Complaint logged via Email'},
                        {'timestamp': datetime.datetime(2024, 8, 31, 5, 56),
                         'status': 'InProgress',
                         'actor': 'Agent',
                         'note': 'Assigned to support agent'},
                        {'timestamp': datetime.datetime(2024, 9, 1, 5, 56),
                         'status': 'Resolved',
                         'actor': 'Agent',
                         'note': 'Resolved wi

### 3.3 `customer_cases` — the integrated view

We embed orders - deliveries - incidents and a complaint list, then pre-compute a `summary` block.

In [12]:
# Build look-up indexes for the embed pass
orders_by_cust       = {k: g for k, g in tables['orders'].groupby('customer_id')}
deliveries_by_order  = {r['order_id']: r for _, r in tables['deliveries'].iterrows()}
incidents_by_delivery = {}
for _, r in tables['incidents'].iterrows():
    incidents_by_delivery.setdefault(r['delivery_id'], []).append(r)
complaints_by_cust   = {k: g for k, g in tables['complaints'].groupby('customer_id')}


def to_customer_doc(cust):
    cid = cust['customer_id']

    cust_orders = []
    for _, o in orders_by_cust.get(cid, pd.DataFrame()).iterrows():
        delivery_row = deliveries_by_order.get(o['order_id'])
        delivery_doc = None
        if delivery_row is not None:
            d_inc = incidents_by_delivery.get(delivery_row['delivery_id'], [])
            delivery_doc = {
                'delivery_id':                   delivery_row['delivery_id'],
                'driver_id':                     delivery_row['driver_id'],
                'vehicle_id':                    delivery_row['vehicle_id'],
                'hub_id':                        delivery_row['hub_id'],
                'dispatch_time':                 delivery_row['dispatch_time'].to_pydatetime()
                                                 if pd.notna(delivery_row['dispatch_time']) else None,
                'delivery_status':               delivery_row['delivery_status'],
                'manual_route_override_count':   int(delivery_row['manual_route_override_count']),
                'customer_rating_post_delivery': float(delivery_row['customer_rating_post_delivery'])
                                                 if pd.notna(delivery_row['customer_rating_post_delivery']) else None,
                'fuel_or_charge_cost':           float(delivery_row['fuel_or_charge_cost']),
                'incidents': [{
                    'incident_id':       inc['incident_id'],
                    'incident_type':     inc['incident_type'],
                    'severity':          inc['severity'],
                    'resolution_status': inc['resolution_status'],
                } for inc in d_inc],
            }
        cust_orders.append({
            'order_id':         o['order_id'],
            'service_type':     o['service_type'],
            'order_created_at': o['order_created_at'].to_pydatetime(),
            'order_value':      float(o['order_value']),
            'pickup_zone':      o['pickup_zone'],
            'dropoff_zone':     o['dropoff_zone'],
            'delivery':         delivery_doc,
        })

    cust_complaints = []
    for _, c in complaints_by_cust.get(cid, pd.DataFrame()).iterrows():
        cust_complaints.append({
            'complaint_id':   c['complaint_id'],
            'order_id':       c['order_id'] if pd.notna(c['order_id']) else None,
            'complaint_type': c['complaint_type'],
            'severity':       c['severity'],
            'status':         c['status'],
            'created_at':     c['created_at'].to_pydatetime(),
        })

    return {
        '_id': cid,
        'profile': {
            'age':               int(cust['age']),
            'home_zone':         cust['home_zone'],
            'customer_type':     cust['customer_type'],
            'signup_date':       cust['signup_date'].to_pydatetime(),
            'loyalty_score':     float(cust['loyalty_score']) if pd.notna(cust['loyalty_score']) else None,
            'preferred_channel': cust['preferred_channel'] if pd.notna(cust['preferred_channel']) else None,
            'account_status':    cust['account_status'],
        },
        'orders':     cust_orders,
        'complaints': cust_complaints,
        'summary': {
            'total_orders':     len(cust_orders),
            'total_complaints': len(cust_complaints),
            'total_value':      float(sum(o['order_value'] for o in cust_orders)),
        }
    }


customer_docs = [to_customer_doc(r) for _, r in tables['customers'].iterrows()]
db['customer_cases'].drop()
db['customer_cases'].insert_many(customer_docs)
print(f"customer_cases: inserted {db['customer_cases'].count_documents({})} documents")

# Show a customer that has orders, deliveries and complaints
big = db['customer_cases'].find_one({'summary.total_complaints': {'$gte': 2}})
if big:
    print(f"\nIntegrated view for customer {big['_id']}: "
          f"{big['summary']['total_orders']} orders, "
          f"{big['summary']['total_complaints']} complaints, "
          f"£{big['summary']['total_value']:.2f} lifetime value")


customer_cases: inserted 650 documents

Integrated view for customer C0001: 3 orders, 2 complaints, £332.23 lifetime value


## 4. Read operations — `find` and projections

### 4.1 Basic filters and projections

In [13]:
# Q1. High-severity complaints still open, sorted newest first
for c in db['complaint_cases'].find(
        {'severity': 'High', 'status': {'$ne': 'Resolved'}},
        {'customer_id': 1, 'complaint_type': 1, 'channel': 1, 'created_at': 1}
    ).sort('created_at', -1).limit(5):
    print(c)


{'_id': 'CP0287', 'customer_id': 'C0405', 'complaint_type': 'DriverBehaviour', 'channel': 'Phone', 'created_at': datetime.datetime(2025, 12, 27, 15, 30)}
{'_id': 'CP0107', 'customer_id': 'C0396', 'complaint_type': 'Delay', 'channel': 'Email', 'created_at': datetime.datetime(2025, 12, 23, 16, 37)}
{'_id': 'CP0131', 'customer_id': 'C0583', 'complaint_type': 'Delay', 'channel': 'Email', 'created_at': datetime.datetime(2025, 12, 9, 9, 35)}
{'_id': 'CP0130', 'customer_id': 'C0581', 'complaint_type': 'Delay', 'channel': 'Email', 'created_at': datetime.datetime(2025, 11, 27, 10, 42)}
{'_id': 'CP0063', 'customer_id': 'C0164', 'complaint_type': 'Billing', 'channel': 'App', 'created_at': datetime.datetime(2025, 6, 28, 9, 52)}


In [14]:
# Q2. Sessions that took place on Android with at least one chat event
# Uses dot-notation to query inside the embedded array.
n = db['app_sessions'].count_documents({
    'device_type': 'Android',
    'events.event_type': {'$in': ['chat_opened', 'chat_escalated']}
})
print(f"Android sessions with a chat event: {n}")


Android sessions with a chat event: 63


### 4.2 The integrated view — the board's actual question, in one query

> *"Give me every fact about this customer."*

In [15]:
# Q3. ONE find_one() returns the complete view that previously required
# 4 SQL joins and an application-side stitch-up.
cust = db['customer_cases'].find_one({'_id': 'C0292'})
if cust:
    print(f"Customer: {cust['_id']}")
    print(f"  Profile: {cust['profile']}")
    print(f"  Summary: {cust['summary']}")
    print(f"  First order: {cust['orders'][0] if cust['orders'] else 'none'}")
    print(f"  Complaints: {cust['complaints']}")


Customer: C0292
  Profile: {'age': 24, 'home_zone': 'South', 'customer_type': 'Consumer', 'signup_date': datetime.datetime(2025, 3, 2, 11, 24), 'loyalty_score': 73.2, 'preferred_channel': 'App', 'account_status': 'Active'}
  Summary: {'total_orders': 4, 'total_complaints': 1, 'total_value': 395.9}
  First order: {'order_id': 'O00001', 'service_type': 'Passenger', 'order_created_at': datetime.datetime(2024, 8, 20, 14, 43), 'order_value': 126.65, 'pickup_zone': 'Airport', 'dropoff_zone': 'South', 'delivery': {'delivery_id': 'DL00937', 'driver_id': 'D047', 'vehicle_id': 'V090', 'hub_id': 'H01', 'dispatch_time': datetime.datetime(2024, 8, 20, 16, 29), 'delivery_status': 'OnTime', 'manual_route_override_count': 2, 'customer_rating_post_delivery': 4.29, 'fuel_or_charge_cost': 15.82, 'incidents': []}}
  Complaints: [{'complaint_id': 'CP0226', 'order_id': 'O01129', 'complaint_type': 'Delay', 'severity': 'Low', 'status': 'Resolved', 'created_at': datetime.datetime(2025, 3, 26, 15, 4)}]


### 4.3 Aggregation framework

Where SQL had `GROUP BY` and joins, MongoDB has a pipeline of stages: `$match`, `$group`, `$sort`, `$project`, `$lookup`, `$unwind`. Each stage feeds the next.

In [16]:
# Q4. Complaint volume by severity x status (the equivalent of a SQL GROUP BY)
pipe = [
    {'$group': {'_id': {'severity': '$severity', 'status': '$status'},
                'n': {'$sum': 1}}},
    {'$sort': {'_id.severity': 1, 'n': -1}}
]
print("Complaints by severity x status:")
for d in db['complaint_cases'].aggregate(pipe):
    print(f"  {d['_id']['severity']:8s} / {d['_id']['status']:18s}  n={d['n']}")


Complaints by severity x status:
  High     / Resolved            n=48
  High     / Open                n=14
  High     / Escalated           n=8
  High     / AwaitingCustomer    n=7
  Low      / Resolved            n=40
  Low      / Open                n=14
  Low      / Escalated           n=9
  Low      / AwaitingCustomer    n=8
  Medium   / Resolved            n=98
  Medium   / Open                n=28
  Medium   / AwaitingCustomer    n=25
  Medium   / Escalated           n=21


In [17]:
# Q5. Unwind the embedded events and find success rate per event_type.
# $unwind is essential when reporting on items inside an array — it produces
# one row per array element, which $group can then aggregate.
pipe = [
    {'$unwind': '$events'},
    {'$group': {'_id': '$events.event_type',
                'n': {'$sum': 1},
                'success_rate': {'$avg': '$events.success_flag'},
                'avg_latency_ms': {'$avg': '$events.api_latency_ms'}}},
    {'$sort': {'success_rate': 1}}
]
print("Event-type performance (worst first):")
for d in db['app_sessions'].aggregate(pipe):
    print(f"  {d['_id']:28s}  n={d['n']:4d}  success={d['success_rate']:.3f}  "
          f"latency={d['avg_latency_ms']:.0f}ms")


Event-type performance (worst first):
  chat_escalated                n=  38  success=0.500  latency=478ms
  payment_retry                 n=  69  success=0.725  latency=473ms
  eta_refresh                   n= 105  success=1.000  latency=452ms
  cancel_attempt                n=  28  success=1.000  latency=417ms
  search_route                  n=  99  success=1.000  latency=457ms
  chat_opened                   n=  88  success=1.000  latency=478ms
  track_order                   n= 138  success=1.000  latency=461ms
  delivery_instruction_update   n=  75  success=1.000  latency=496ms


**Finding worth quoting.** `chat_escalated` events have a **success rate of ~0.50** — only half of escalation attempts succeed. That is a concrete platform problem on top of the customer-experience director's broader complaint about fragmented data. The case study mentioned *"customer complaint histories, service chat transcripts"* among the data that needed better handling — this aggregation surfaces a specific quality issue inside that stream.

In [18]:
# Q6. Top repeat-complainers from the integrated view, with their full context
# in one collection scan. Equivalent SQL would join customers + complaints + group + filter.
pipe = [
    {'$match': {'summary.total_complaints': {'$gte': 2}}},
    {'$project': {
        '_id': 1,
        'customer_type': '$profile.customer_type',
        'home_zone': '$profile.home_zone',
        'total_orders':     '$summary.total_orders',
        'total_complaints': '$summary.total_complaints',
        'total_value':      '$summary.total_value'
    }},
    {'$sort': {'total_complaints': -1, 'total_value': -1}},
    {'$limit': 10}
]
print("Top repeat complainers:")
for d in db['customer_cases'].aggregate(pipe):
    print(f"  {d['_id']}  {d['customer_type']:10s}  {d['home_zone']:10s}  "
          f"complaints={d['total_complaints']}  orders={d['total_orders']}  "
          f"value=£{d['total_value']:.2f}")


Top repeat complainers:
  C0368  Consumer    North       complaints=4  orders=3  value=£235.92
  C0545  Consumer    South       complaints=3  orders=6  value=£923.45
  C0372  Consumer    West        complaints=3  orders=6  value=£669.11
  C0172  Consumer    North       complaints=3  orders=4  value=£575.84
  C0421  Consumer    Central     complaints=3  orders=3  value=£331.91
  C0626  Consumer    South       complaints=3  orders=3  value=£306.66
  C0110  Consumer    East        complaints=3  orders=3  value=£304.09
  C0242  Consumer    East        complaints=3  orders=5  value=£264.86
  C0282  Consumer    Riverside   complaints=3  orders=2  value=£247.88
  C0142  Consumer    South       complaints=3  orders=2  value=£211.00


## 5. Update operations

Two patterns NorthStar will need every day: appending to a resolution history (`$push`) and bulk-flagging records that meet a condition (`update_many` + `$set`).

In [19]:
# U1. Push a new event onto an existing complaint's resolution_history.
# This is the canonical pattern for the case study's 'long sequences of messages,
# repeated status changes, escalation notes'.
result = db['complaint_cases'].update_one(
    {'_id': 'CP0001'},
    {
        '$set': {'priority_review': True,
                 'last_touched': datetime.now()},
        '$push': {'resolution_history': {
            'timestamp': datetime.now(),
            'status':    'Escalated',
            'actor':     'System',
            'note':      'Flagged for manager review after SLA breach'
        }}
    }
)
print(f"update_one CP0001: matched={result.matched_count}, modified={result.modified_count}")

# Verify
updated = db['complaint_cases'].find_one({'_id': 'CP0001'},
                                          {'resolution_history': 1, 'priority_review': 1})
print(f"  Now has {len(updated['resolution_history'])} history entries.")
print(f"  priority_review = {updated.get('priority_review')}")


update_one CP0001: matched=1, modified=1
  Now has 2 history entries.
  priority_review = True


In [20]:
# U2. Bulk-flag all High-severity Open complaints for SLA tracking.
result = db['complaint_cases'].update_many(
    {'severity': 'High', 'status': 'Open'},
    {'$set': {'sla_clock_started': True,
              'sla_target_days':   7}}      # business rule: high-sev must close in 7d
)
print(f"update_many: matched={result.matched_count}, modified={result.modified_count}")


update_many: matched=14, modified=14


In [21]:
# U3. Update inside the integrated customer doc — append a complaint summary
# (the kind of write the customer-service portal would do).
# $push with $each lets us add multiple items in one call.
result = db['customer_cases'].update_one(
    {'_id': 'C0001'},
    {'$inc': {'summary.total_complaints': 1},
     '$push': {'complaints': {
        'complaint_id': 'CP_NEW_001',
        'order_id':     None,
        'complaint_type': 'AppIssue',
        'severity': 'Low',
        'status':   'Open',
        'created_at': datetime.now()
     }}}
)
print(f"customer_cases update: modified={result.modified_count}")
print("Confirming the new complaint is now in the integrated view:")
print(db['customer_cases'].find_one(
    {'_id': 'C0001'},
    {'complaints': {'$slice': -1}, 'summary': 1})
)


customer_cases update: modified=1
Confirming the new complaint is now in the integrated view:
{'_id': 'C0001', 'complaints': [{'complaint_id': 'CP_NEW_001', 'order_id': None, 'complaint_type': 'AppIssue', 'severity': 'Low', 'status': 'Open', 'created_at': datetime.datetime(2026, 5, 21, 12, 23, 4, 545000)}], 'summary': {'total_orders': 3, 'total_complaints': 3, 'total_value': 332.23}}


## 6. Delete operations

GDPR-style "right to erasure" requests are the most common production use of `delete`. The two patterns below cover the realistic cases.

In [22]:
# D1. Insert a test doc, then delete it (so we don't damage real data)
db['complaint_cases'].insert_one({
    '_id': 'CP_TEST_DELETE', 'severity': 'Low', 'status': 'Resolved'
})
result = db['complaint_cases'].delete_one({'_id': 'CP_TEST_DELETE'})
print(f"delete_one: deleted={result.deleted_count}")

# D2. Selectively pull an item out of an embedded array
# ($pull is the inverse of $push - removes elements matching a condition).
demo = db['customer_cases'].update_one(
    {'_id': 'C0001'},
    {'$pull': {'complaints': {'complaint_id': 'CP_NEW_001'}},
     '$inc': {'summary.total_complaints': -1}}
)
print(f"$pull from embedded array: modified={demo.modified_count}")


delete_one: deleted=1
$pull from embedded array: modified=1


## 7. Validate the loaded database

In [23]:
# Final snapshot
for c in ['app_sessions', 'complaint_cases', 'customer_cases']:
    n = db[c].count_documents({})
    one = db[c].find_one()
    keys = sorted(one.keys()) if one else []
    print(f"{c:20s}  n={n:4d}  fields={keys}")


app_sessions          n= 637  fields=['_id', 'customer_id', 'device_type', 'event_count', 'events', 'session_end', 'session_start', 'zone_context']
complaint_cases       n= 320  fields=['_id', 'channel', 'compensation_amount', 'complaint_type', 'created_at', 'customer_id', 'last_touched', 'order_id', 'priority_review', 'resolution_days', 'resolution_history', 'severity', 'sla_clock_started', 'sla_target_days', 'status']
customer_cases        n= 650  fields=['_id', 'complaints', 'orders', 'profile', 'summary']


## 8. Summary — what this notebook delivered against the rubric

| Rubric criterion | Where in this notebook |
|---|---|
| PyMongo / integration with Python | §1 (connection), §3 (builders that go pandas → dict → Mongo), §4–§6 (every operation) |
| CRUD operations | §3 (Create / `insert_many`), §4 (Read / `find` + `aggregate`), §5 (Update / `update_one`, `update_many`, `$push`, `$set`, `$inc`, `$pull`), §6 (Delete / `delete_one`, `$pull`) |
| Appropriate NoSQL data modelling | §2 — explicit rationale for what stays relational, what becomes documents, what gets embedded vs referenced; three collections each justified against the case study |
| Aggregation framework | §4.3 — `$group`, `$sort`, `$unwind`, `$match`, `$project` |
| Design and implementation of MongoDB database | §2 (design) + §3 (build) |

### Business findings produced by this notebook

1. **`chat_escalated` succeeds only ~50% of the time** — a concrete platform-level failure mode that complaint and operational data alone could not surface.
2. **The integrated `customer_cases` collection collapses the board's four-table question to a single read.** That, structurally, is what the case study asks for in its closing paragraphs.
3. **High-severity complaints now carry a `sla_clock_started` flag** (§5.2) — the schema can support the triage-rule recommendation that emerged from the customer-experience director's hypothesis.

### Handover to `06_indexing.ipynb`

The next notebook adds **indexes** to support the highest-frequency queries from §4, measures the speed-up with `explain('executionStats')`, and justifies each index choice. The four queries that most benefit:

- `complaint_cases.find({'severity': ..., 'status': ...})` → compound index on `(severity, status)`
- `customer_cases.find({'_id': ...})` → already indexed (`_id` is automatic)
- `app_sessions.find({'events.event_type': ...})` → multikey index on `events.event_type`
- `complaint_cases.find({'customer_id': ...})` → single-field index
